## Comprehensive Model Evaluation on Test Data
 
### This notebook loads all previously trained models and evaluates them on the master test set (`data/master_splits/test_set.parquet`). It performs the following steps for each model:

### 1.  **Loads the required test data and labels.**
### 2.  **Loads the trained model artifacts** from the `models/` directory.
### 3.  **Generates predictions** on the unseen test data.
### 4.  **Calculates and displays key performance metrics** (Classification Report, Accuracy, ROC-AUC, etc.).
### 5.  **Generates and saves all evaluation plots** (Confusion Matrix, Feature Importance, etc.) to the `results/` directory.

### This provides a complete, end-to-end performance summary of the entire trained pipeline.

In [1]:
# ===================================================================
# CELL 1: SETUP AND IMPORTS
# ===================================================================
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder

# --- Import all project-specific classes and configurations ---
# This assumes the notebook is in the root of your project directory.
from config import Config
from data_loader import (
    get_data_for_binary, get_data_for_multiclass, get_data_for_virus,
    load_master_test_set, IoTDataset
)
from binary_classifier import BinaryClassifier
from multiclass_classifier import MultiClassXGBoost, MultiClassNeuralNetModel
from virus_classifier import VirusClassifier
from autoencoder import AutoencoderModel
from autoencoder_denoising import DenoisingAutoencoderModel
from clustering import KMeansClustering

# Ensure the results directory exists
os.makedirs(Config.RESULTS_DIR, exist_ok=True)

# Set seeds and print mode info for consistency
Config.set_seeds()
Config.print_mode_info()

# ## 1. Binary Classification: Benign vs. Malicious
# 
# Here, we evaluate the performance of the tuned LightGBM model and the baseline logistic regression model on the binary task of identifying traffic as either benign or malicious.



⚙️  CONFIGURATION MODE
   Mode: PRODUCTION MODE 🚀
   Raw Data Directory: data/raw/
   Device: cuda


In [2]:
# %%
# ===================================================================
# CELL 2: EVALUATE BINARY CLASSIFIERS
# ===================================================================
print("=" * 70)
print("Evaluating Binary Classification Models")
print("=" * 70)

# 1. Load Data
# The function conveniently splits the master set for us. We only need the test portion.
_, X_test_bin, _, y_test_bin = get_data_for_binary()

# 2. Evaluate Tuned LightGBM Model
print("\n--- Evaluating: Tuned LightGBM ---")
try:
    lgbm_classifier = BinaryClassifier()
    model_path = os.path.join(Config.MODELS_DIR, 'binary_classifier_tuned.pkl')
    model_data = joblib.load(model_path)
    lgbm_classifier.model = model_data['model']
    lgbm_classifier.best_params = model_data['params']
    
    print(f"✓ Model loaded from {model_path}")
    lgbm_results = lgbm_classifier.evaluate(X_test_bin, y_test_bin)
except FileNotFoundError:
    print(f"❌ ERROR: Model file not found at {model_path}. Please run binary_classifier.py first.")

# 3. Evaluate Baseline Logistic Regression Model
if Config.RUN_LOGISTIC_REGRESSION:
    print("\n--- Evaluating: Baseline LightGBM (Logistic Config) ---")
    try:
        lr_classifier = BinaryClassifier()
        model_path = os.path.join(Config.MODELS_DIR, 'binary_classifier_logistic.pkl')
        model_data = joblib.load(model_path)
        
        # =================================================================
        # THIS IS THE CORRECTED LINE:
        lr_classifier.model = model_data['model'] 
        # =================================================================
        
        print(f"✓ Model loaded from {model_path}")
        # Pass save_plots=False to avoid overwriting the main model's plots if names are the same
        # For this project, plot names are distinct, so it's safe to set to True.
        lr_results = lr_classifier.evaluate(X_test_bin, y_test_bin, save_plots=True) 
    except FileNotFoundError:
        print(f"❌ ERROR: Model file not found at {model_path}. Please run binary_classifier.py first.")

Evaluating Binary Classification Models

🔧 PREPARING BINARY CLASSIFICATION DATA

📂 Loading MASTER TRAIN set from: data/master_splits/train_set.parquet
✓ Loaded 39,999,996 training samples.

📂 Loading MASTER TEST set from: data/master_splits/test_set.parquet
✓ Loaded 10,000,000 test samples.
✓ Binary data ready.
  - Train samples: 39,999,996
  - Test samples:  10,000,000

--- Evaluating: Tuned LightGBM ---

ADVANCED MODEL: Tuned LightGBM
Binary Classifier (LightGBM) initialized
  Using Optuna: True
  Using SMOTE: True
✓ Model loaded from models/binary_classifier_tuned.pkl

📊 EVALUATING ADVANCED LightGBM MODEL
✓ Predictions complete (18.11s)

──────────────────────────────────────────────────────────────────────
CLASSIFICATION REPORT
──────────────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

      Benign       0.01      0.99      0.03       365
   Malicious       1.00      1.00      1.00   9999635

    accuracy           

In [3]:
# %%
# ===================================================================
# CELL: CORRECTED DIAGNOSTIC - INSPECT SAVED MODEL FILE
# ===================================================================
import torch
import os
from config import Config

print("--- Inspecting the contents of multiclass_nn.pth ---")

try:
    # --- Define the path to the model file ---
    model_filename = 'multiclass_nn.pth'
    model_path = os.path.join(Config.MODELS_DIR, model_filename)

    # --- Load the saved checkpoint directly WITH THE FIX ---
    # This will now successfully load the file.
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

    # --- Print all the keys found in the saved file ---
    print("\nKeys found in the file:", list(checkpoint.keys()))

    # --- Check specifically for the 'scaler' key ---
    if 'scaler' in checkpoint:
        print("\n✅ The 'scaler' key EXISTS in the file.")
        
        if checkpoint['scaler'] is not None:
            print("   - The scaler object is NOT None. It is a valid object.")
            print("\n   >>> CONCLUSION: The saved model file is CORRECT. <<<")
        else:
            print("   - ❌ The scaler object is None inside the file.")
            
    else:
        print("\n❌ The 'scaler' key DOES NOT EXIST in the file.")

except FileNotFoundError:
    print(f"\n❌ ERROR: The model file was not found at {model_path}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

--- Inspecting the contents of multiclass_nn.pth ---

Keys found in the file: ['model_state', 'scaler', 'num_classes', 'hidden_layers', 'dropout_rate', 'learning_rate', 'use_batch_norm', 'best_params']

✅ The 'scaler' key EXISTS in the file.
   - The scaler object is NOT None. It is a valid object.

   >>> CONCLUSION: The saved model file is CORRECT. <<<


In [3]:
# %%
# ===================================================================
# CELL 3: EVALUATE MULTI-CLASS CLASSIFIERS (FINAL WORKING VERSION)
# ===================================================================
print("\n" + "=" * 70)
print("Evaluating Multi-Class Classification Models")
print("=" * 70)

# 1. Load Data
_, X_test_multi, _, y_test_multi, le_multi = get_data_for_multiclass()
print(f"\n✓ Loaded {len(X_test_multi):,} test samples for multi-class evaluation.")

# 2. Evaluate XGBoost Model
print("\n--- Evaluating: Multi-Class XGBoost ---")
try:
    xgb_model = MultiClassXGBoost()
    model_path = os.path.join(Config.MODELS_DIR, 'multiclass_xgboost.pkl')
    model_data = joblib.load(model_path)
    xgb_model.model = model_data['model']
    xgb_model.best_params = model_data['params']
    
    print(f"✓ Model loaded from {model_path}")
    xgb_results = xgb_model.evaluate(X_test_multi, y_test_multi, le_multi)
except Exception as e:
    print(f"❌ ERROR evaluating XGBoost: {e}")
    
# 3. Evaluate Neural Network Model
print("\n--- Evaluating: Multi-Class Neural Network ---")
try:
    input_dim = X_test_multi.shape[1]
    # Get the number of classes from the label encoder we just loaded
    num_classes = len(le_multi.classes_)
    
    # ============================================================================
    # THIS IS THE FIX: We must now provide 'num_classes' when initializing the model.
    nn_model = MultiClassNeuralNetModel(input_dim, num_classes)
    # ============================================================================
    
    # Define the final, official model filename to load.
    model_filename = 'multiclass_nn.pth' 
    
    # This robust function will read the architecture from the file,
    # build the model correctly, and then load the weights.
    nn_model.load_model(model_filename) 
    
    model_path_for_print = os.path.join(Config.MODELS_DIR, model_filename)
    print(f"\n✓ Notebook is now evaluating model from: {model_path_for_print}")
    
    # The NN model's .evaluate() method expects numpy arrays.
    nn_results = nn_model.evaluate(X_test_multi.values, y_test_multi, le_multi)

except Exception as e:
    # This will catch FileNotFoundError or any other issue during loading/evaluation.
    print(f"❌ AN ERROR OCCURRED: {e}")


Evaluating Multi-Class Classification Models

🔧 PREPARING MULTI-CLASS CLASSIFICATION DATA

📂 Loading MASTER TRAIN set from: data/master_splits/train_set.parquet
✓ Loaded 39,999,996 training samples.

📂 Loading MASTER TEST set from: data/master_splits/test_set.parquet
✓ Loaded 10,000,000 test samples.

🎯 Performing class-aware undersampling on the TRAINING set...
   Original train size: 39,999,996 rows
   New target train size: 1,599,999 rows
✓ Undersampling complete. Final training set size: 1,599,999
✓ Label encoder saved.

📊 Final class distribution (training set):
   Attack: 7,084 (0.44%)
   Benign: 178,763 (11.17%)
   C&C: 12,338 (0.77%)
   DDoS: 63,696 (3.98%)
   FileDownload: 15 (0.00%)
   Okiru: 263,020 (16.44%)
   PortScan: 1,075,065 (67.19%)
   Torii: 18 (0.00%)

✓ Multiclass data ready.
  - Train samples: 1,599,999
  - Test samples:  10,000,000

✓ Loaded 10,000,000 test samples for multi-class evaluation.

--- Evaluating: Multi-Class XGBoost ---
Multi-Class XGBoost initializ

In [7]:
# ===================================================================
# CELL 4: EVALUATE VIRUS FAMILY CLASSIFIER
# ===================================================================
print("\n" + "=" * 70)
print("Evaluating Virus Family Classification Model")
print("=" * 70)

# 1. Load Data
_, X_test_virus, _, y_test_virus, le_virus = get_data_for_virus()
print(f"\n✓ Loaded {len(X_test_virus):,} malicious test samples for virus family evaluation.")

# 2. Evaluate LightGBM Model
try:
    virus_classifier = VirusClassifier()
    model_path = os.path.join(Config.MODELS_DIR, 'virus_classifier.pkl')
    model_data = joblib.load(model_path)
    virus_classifier.model = model_data['model']
    virus_classifier.best_params = model_data['params']
    
    print(f"✓ Model loaded from {model_path}")
    virus_results = virus_classifier.evaluate(X_test_virus, y_test_virus, le_virus)
    
    print("\n--- Analyzing prediction confidence per family ---")
    virus_classifier.analyze_families(X_test_virus, y_test_virus, le_virus)

except FileNotFoundError:
    print(f"❌ ERROR: Model file not found at {model_path}. Please run virus_classifier.py first.")

# ## 4. Anomaly Detection: Autoencoders
# 
# Here, we test our unsupervised autoencoder models. The goal is to see how well they distinguish between benign and malicious traffic by measuring reconstruction error. A good model should have a low detection rate for benign traffic (few false positives) and a high detection rate for malicious traffic (high true positives).


Evaluating Virus Family Classification Model

🔧 PREPARING VIRUS CLASSIFICATION DATA

📂 Loading MASTER TRAIN set from: data/master_splits/train_set.parquet
✓ Loaded 39,999,996 training samples.

📂 Loading MASTER TEST set from: data/master_splits/test_set.parquet
✓ Loaded 10,000,000 test samples.
✓ Label encoder saved.

✓ Virus data ready.
  - Train samples: 39,998,535
  - Test samples:  9,999,635

✓ Loaded 9,999,635 malicious test samples for virus family evaluation.
Virus/Malware Family Classifier initialized
  Using Optuna: True
  Using SMOTE: True
✓ Model loaded from models/virus_classifier.pkl

📊 EVALUATING VIRUS CLASSIFIER

⏳ Generating predictions...
✓ Predictions complete (66.40s)
   Throughput: 150591 samples/sec

──────────────────────────────────────────────────────────────────────
CLASSIFICATION REPORT
──────────────────────────────────────────────────────────────────────
               precision    recall  f1-score   support

       Gagfyt       1.00      1.00      1.00    

/workspace/malaware_analysis/virus_classifier.py:224: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=importance[indices], y=[feature_names[i] for i in indices], palette='viridis')


  ✓ Saved feature importance plot

⏱️  Total evaluation time: 71.88s

--- Analyzing prediction confidence per family ---

🔬 ANALYZING PREDICTION CONFIDENCE PER FAMILY

📊 Gagfyt:
   Samples: 265,489
   Accuracy: 100.00% (265489/265489)
   Avg Confidence (when correct): 0.9789

📊 Hajime:
   Samples: 216,319
   Accuracy: 99.61% (215472/216319)
   Avg Confidence (when correct): 0.8737

📊 Hakai:
   Samples: 434
   Accuracy: 100.00% (434/434)
   Avg Confidence (when correct): 0.9742

📊 Hide and Seek:
   Samples: 202,416
   Accuracy: 99.74% (201880/202416)
   Avg Confidence (when correct): 0.9759

📊 IRCBot:
   Samples: 2,498,546
   Accuracy: 99.99% (2498410/2498546)
   Avg Confidence (when correct): 0.9782

📊 Kenjiro:
   Samples: 2,081,093
   Accuracy: 68.49% (1425300/2081093)
   Avg Confidence (when correct): 0.8565

📊 Mirai:
   Samples: 4,237,616
   Accuracy: 91.97% (3897135/4237616)
   Avg Confidence (when correct): 0.8961

📊 Muhstik:
   Samples: 31,141
   Accuracy: 100.00% (31141/31141)
 

In [2]:
# %%
# ===================================================================
# CELL 5: EVALUATE ANOMALY DETECTION MODELS (FINAL CORRECTED VERSION)
# ===================================================================
print("\n" + "=" * 70)
print("Evaluating Anomaly Detection (Autoencoder) Models")
print("=" * 70)

# 1. Load the full test set
df_test_full = load_master_test_set()
features_used = joblib.load(Config.AUTOENCODER_FEATURE_LIST_PATH)
X_test_benign_ae = df_test_full[df_test_full[Config.TARGET_COL] == 'Benign'][features_used].fillna(0)
X_test_malicious_ae = df_test_full[df_test_full[Config.TARGET_COL] == 'Malicious'][features_used].fillna(0)
print(f"✓ Loaded {len(X_test_benign_ae):,} benign and {len(X_test_malicious_ae):,} malicious test samples.")

# --- Evaluate Standard Autoencoder ---
print("\n--- Evaluating: Standard Autoencoder ---")
try:
    input_dim_ae = len(features_used)
    autoencoder = AutoencoderModel(input_dim_ae)
    
    # This robust function will now rebuild the architecture from the file
    autoencoder.load_model('autoencoder_final.pth')
    
    # Scale data using the scaler that was just loaded
    X_benign_scaled = autoencoder.scaler.transform(X_test_benign_ae.values)
    X_malicious_scaled = autoencoder.scaler.transform(X_test_malicious_ae.values)
    
    benign_loader = DataLoader(IoTDataset(X_benign_scaled), batch_size=Config.AUTOENCODER_BATCH_SIZE)
    malicious_loader = DataLoader(IoTDataset(X_malicious_scaled), batch_size=Config.AUTOENCODER_BATCH_SIZE)

    print("\n--- Results on BENIGN Data (Standard AE) ---")
    benign_preds, _ = autoencoder.detect_anomalies(benign_loader)
    false_positive_rate = benign_preds.sum() / len(benign_preds) * 100 if len(benign_preds) > 0 else 0
    print(f"  🚨 False Positive Rate: {false_positive_rate:.2f}% ({benign_preds.sum():,}/{len(benign_preds):,})")

    print("\n--- Results on MALICIOUS Data (Standard AE) ---")
    malicious_preds, _ = autoencoder.detect_anomalies(malicious_loader)
    detection_rate = malicious_preds.sum() / len(malicious_preds) * 100 if len(malicious_preds) > 0 else 0
    print(f"  🎯 Detection Rate: {detection_rate:.2f}% ({malicious_preds.sum():,}/{len(malicious_preds):,})")

except Exception as e:
     print(f"❌ ERROR evaluating Standard Autoencoder: {e}")

# --- Evaluate Denoising Autoencoder ---
print("\n--- Evaluating: Denoising Autoencoder ---")
try:
    input_dim_dae = len(features_used)
    denoising_ae = DenoisingAutoencoderModel(input_dim_dae)
    
    # This robust function will now raise an error if the scaler is missing
    denoising_ae.load_model('denoising_autoencoder_final.pth')

    # Scale data using the scaler that was just loaded
    X_benign_scaled_dae = denoising_ae.scaler.transform(X_test_benign_ae.values)
    X_malicious_scaled_dae = denoising_ae.scaler.transform(X_test_malicious_ae.values)

    benign_loader_dae = DataLoader(IoTDataset(X_benign_scaled_dae), batch_size=Config.AUTOENCODER_BATCH_SIZE)
    malicious_loader_dae = DataLoader(IoTDataset(X_malicious_scaled_dae), batch_size=Config.AUTOENCODER_BATCH_SIZE)

    print("\n--- Results on BENIGN Data (Denoising AE) ---")
    benign_preds_dae, _ = denoising_ae.detect_anomalies(benign_loader_dae)
    fp_rate_dae = benign_preds_dae.sum() / len(benign_preds_dae) * 100 if len(benign_preds_dae) > 0 else 0
    print(f"  🚨 False Positive Rate: {fp_rate_dae:.2f}% ({benign_preds_dae.sum():,}/{len(benign_preds_dae):,})")

    print("\n--- Results on MALICIOUS Data (Denoising AE) ---")
    malicious_preds_dae, _ = denoising_ae.detect_anomalies(malicious_loader_dae)
    detection_rate_dae = malicious_preds_dae.sum() / len(malicious_preds_dae) * 100 if len(malicious_preds_dae) > 0 else 0
    print(f"  🎯 Detection Rate: {detection_rate_dae:.2f}% ({malicious_preds_dae.sum():,}/{len(malicious_preds_dae):,})")

except Exception as e:
     print(f"❌ ERROR evaluating Denoising Autoencoder: {e}")


Evaluating Anomaly Detection (Autoencoder) Models

📂 Loading MASTER TEST set from: data/master_splits/test_set.parquet
✓ Loaded 10,000,000 test samples.
✓ Loaded 365 benign and 9,999,635 malicious test samples.

--- Evaluating: Standard Autoencoder ---
Autoencoder initialized on cuda
💾 Reading architecture from checkpoint...

🏗️  Building model with architecture: [128, 64, 22], latent_dim: 11
✅ Model state loaded successfully into matching architecture.

--- Results on BENIGN Data (Standard AE) ---


Processing batches: 100%|██████████| 1/1 [00:00<00:00,  4.66it/s]


  🚨 False Positive Rate: 13.97% (51/365)

--- Results on MALICIOUS Data (Standard AE) ---


Processing batches: 100%|██████████| 306/306 [00:50<00:00,  6.12it/s]


  🎯 Detection Rate: 99.18% (9,917,953/9,999,635)

--- Evaluating: Denoising Autoencoder ---
Denoising Autoencoder initialized on cuda
❌ ERROR evaluating Denoising Autoencoder: ❌ Model file models/denoising_autoencoder_final.pth not found


In [ ]:
# ## 5. Clustering Visualization
# 
# Finally, we apply the trained K-Means model to the malicious traffic in the test set. We then visualize the resulting clusters and color the points by their true malware family labels. This helps us see if the unsupervised clustering algorithm was able to find meaningful, underlying groups in the data that correspond to the actual malware families.
# %%
# ===================================================================
# CELL 6: VISUALIZE CLUSTERING ON TEST DATA
# ===================================================================
print("\n" + "=" * 70)
print("Visualizing K-Means Clustering on Test Data")
print("=" * 70)

try:
    # 1. Load the clustering model
    kmeans_model = KMeansClustering()
    model_path_kmeans = os.path.join(Config.MODELS_DIR, 'kmeans_clustering.pkl')
    kmeans_model.load_model(model_path_kmeans)
    print(f"✓ Clustering model loaded from {model_path_kmeans}")

    # 2. Prepare malicious data from the test set
    df_test_full = load_master_test_set()
    malicious_test_df = df_test_full[df_test_full[Config.TARGET_COL] == 'Malicious'].copy()
    
    # Use the features the model was trained on
    X_malicious_test = malicious_test_df[kmeans_model.feature_names_in_]

    # 3. Get true labels and encode them
    true_labels_str = malicious_test_df[Config.FAMILY_TARGET_COL].fillna('Unknown').astype(str)
    le_virus_viz = LabelEncoder() # Use a new encoder just for this viz
    true_labels_encoded = le_virus_viz.fit_transform(true_labels_str)
    label_names = le_virus_viz.classes_.tolist()

    print(f"\n✓ Prepared {len(X_malicious_test):,} malicious test samples for visualization.")
    
    # 4. Assign test data to clusters
    cluster_labels = kmeans_model.predict(X_malicious_test)
    kmeans_model.labels = cluster_labels # Set the labels on the object for the visualizer to use

    # 5. Generate and save the visualization
    kmeans_model.visualize_clusters_by_label(
        X_malicious_test,
        true_labels_encoded,
        label_names=label_names,
        save_plot=True
    )
    print("\n✓ Clustering visualization complete and saved to the results directory.")
    
except FileNotFoundError:
    print(f"❌ ERROR: Model file not found at {model_path_kmeans}. Please run clustering.py first.")

# %% [markdown]
# ---
# ## Evaluation Complete
# 
# All models have been evaluated on the master test set. All reports have been printed above, and all visualization artifacts have been saved to the `/results` directory.
# ---